# TensorQuantLib v0.3.0 — Interactive Tutorial

This notebook walks through the complete TensorQuantLib workflow:

1. **Autograd Basics** — Build computation graphs and compute gradients
2. **Black-Scholes Pricing** — Price options and compute Greeks via autodiff
3. **Second-Order Greeks** — Gamma, Vanna, Volga via second-order autodiff
4. **TT-SVD Compression** — Compress multi-dimensional tensors
5. **TT-Surrogate Pricing** — Build a full surrogate for basket options
6. **TT-Cross (6+ Assets)** — Scale to 6 assets without full-grid computation
7. **Stochastic Volatility** — Heston model pricing and calibration
8. **Volatility Surfaces** — SABR and SVI parameterizations
9. **Exotics, IR, FX, Credit** — Full product coverage
10. **Visualization** — Plot pricing and Greek surfaces

In [1]:
# Install if needed (uncomment)
# !pip install -e ".[all]"

import numpy as np
import matplotlib.pyplot as plt

import tensorquantlib as tql
from tensorquantlib.core.tensor import Tensor

print(f"TensorQuantLib v{tql.__version__}")

ModuleNotFoundError: No module named 'tensorquantlib'

---
## 1. Autograd Basics

TensorQuantLib includes a custom reverse-mode autodiff engine. Every `Tensor` operation builds a computation graph that can be differentiated.

In [ ]:
# Create tensors with gradient tracking
x = Tensor(np.array([3.0]), requires_grad=True)
y = Tensor(np.array([4.0]), requires_grad=True)

# Build a computation graph: f(x, y) = x^2 * y + y^3
f = x * x * y + y * y * y

print(f"f(3, 4) = {f.item():.1f}")
print(f"Expected: 3^2 * 4 + 4^3 = {3**2 * 4 + 4**3}")

# Backpropagate
f.backward()

print(f"\ndf/dx = {x.grad[0]:.1f}  (expected: 2*x*y = {2*3*4})")
print(f"df/dy = {y.grad[0]:.1f}  (expected: x^2 + 3*y^2 = {3**2 + 3*4**2})")

---
## 2. Black-Scholes Pricing with Autodiff Greeks

Price a European call option and compute Delta (dPrice/dS) automatically.

In [ ]:
from tensorquantlib import bs_price_tensor, bs_price_numpy, bs_delta

# Parameters
S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2

# --- Analytic price ---
price_np = bs_price_numpy(S0, K, T, r, sigma)
delta_analytic = bs_delta(S0, K, T, r, sigma)
print(f"Analytic price:  {price_np:.6f}")
print(f"Analytic Delta:  {delta_analytic:.6f}")

# --- Autodiff price ---
S_t = Tensor(np.array([S0]), requires_grad=True)
price_t = bs_price_tensor(S_t, K=K, T=T, r=r, sigma=sigma)
price_t.backward()

print(f"\nAutodiff price:  {price_t.item():.6f}")
print(f"Autodiff Delta:  {S_t.grad[0]:.6f}")
print(f"\nDelta match: {np.isclose(delta_analytic, S_t.grad[0], atol=1e-6)}")

In [ ]:
# Plot Delta as a function of spot price
spots = np.linspace(60, 140, 200)
deltas = [bs_delta(s, K, T, r, sigma) for s in spots]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(spots, deltas, 'b-', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(K, color='red', linestyle='--', alpha=0.5, label=f'Strike K={K}')
ax.set_xlabel('Spot Price S')
ax.set_ylabel('Delta')
ax.set_title('Black-Scholes Delta vs Spot Price')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Second-Order Greeks — Gamma, Vanna, Volga

TensorQuantLib v0.3.0 introduces the `core.second_order` module with a *hybrid semi-analytic* method: analytical first-order gradients are differentiated once more via central differences, giving ~1e-8 accuracy for Gamma.

| Greek | Meaning | Formula |
|-------|---------|---------|
| **Gamma** | d²Price/dS² | Curvature of Price w.r.t. Spot |
| **Vanna** | d²Price/dS dσ | How Delta changes with Vol |
| **Volga** | d²Price/dσ² | Curvature of Price w.r.t. Vol |

In [ ]:
from tensorquantlib import bs_gamma, bs_price_tensor
from tensorquantlib import gamma_autograd, vanna_autograd, volga_autograd

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2

# --- Gamma ---
gamma_analytic = bs_gamma(S0, K, T, r, sigma)
gamma_auto = gamma_autograd(bs_price_tensor, S0, K, T, r, sigma)

print(f"Gamma (analytic):  {gamma_analytic:.8f}")
print(f"Gamma (autograd):  {gamma_auto:.8f}")
print(f"Relative error:    {abs(gamma_auto - gamma_analytic)/gamma_analytic:.2e}")

# --- Vanna (d²P / dS dσ) ---
vanna = vanna_autograd(bs_price_tensor, S0, K, T, r, sigma)
print(f"\nVanna (autograd):  {vanna:.8f}")

# --- Volga / Vomma (d²P / dσ²) ---
volga = volga_autograd(bs_price_tensor, S0, K, T, r, sigma)
print(f"Volga (autograd):  {volga:.8f}")

In [ ]:
# Plot Gamma surface: accuracy of autograd vs analytic across strikes and spots
spots_g = np.linspace(70, 130, 60)
gammas_ana = [bs_gamma(s, K, T, r, sigma) for s in spots_g]
gammas_auto = [gamma_autograd(bs_price_tensor, s, K, T, r, sigma) for s in spots_g]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(spots_g, gammas_ana, 'b-', linewidth=2, label='Analytic')
ax1.plot(spots_g, gammas_auto, 'r--', linewidth=1.5, label='Autograd', alpha=0.8)
ax1.set_xlabel('Spot S')
ax1.set_ylabel('Gamma')
ax1.set_title('Gamma: Analytic vs Second-Order Autograd')
ax1.legend()
ax1.grid(alpha=0.3)

errors = [abs(a - b) for a, b in zip(gammas_auto, gammas_ana)]
ax2.semilogy(spots_g, errors, 'g-', linewidth=2)
ax2.set_xlabel('Spot S')
ax2.set_ylabel('|Error|')
ax2.set_title('Absolute Error in Gamma (log scale)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Max absolute error: {max(errors):.2e}")

---
## 3. TT-SVD Compression

Compress a multi-dimensional tensor and examine the compression ratio.

In [ ]:
from tensorquantlib import tt_svd, tt_to_full, tt_ranks, tt_memory, tt_error, tt_compression_ratio

# Create a smooth 4D tensor (simulating a pricing surface)
n = 20
x = np.linspace(0, 1, n)
X1, X2, X3, X4 = np.meshgrid(x, x, x, x, indexing='ij')
A = np.exp(-(X1**2 + X2**2 + X3**2 + X4**2))  # Smooth Gaussian

print(f"Original shape: {A.shape}")
print(f"Original size:  {A.nbytes / 1024:.1f} KB")

# Compress with different tolerances
for eps in [1e-2, 1e-4, 1e-8]:
    cores = tt_svd(A, eps=eps)
    ratio = tt_compression_ratio(cores, A.shape)
    error = tt_error(cores, A)
    ranks = tt_ranks(cores)
    print(f"\neps={eps:.0e}: ranks={ranks}, ratio={ratio:.1f}×, error={error:.2e}")

---
## 4. TT-Surrogate for Basket Options

Build a compressed surrogate for a 3-asset basket call option.

In [ ]:
from tensorquantlib import TTSurrogate

# Build a 3-asset basket option surrogate
surr = TTSurrogate.from_basket_analytic(
    S0_ranges=[(80, 120)] * 3,
    K=100, T=1.0, r=0.05,
    sigma=[0.2, 0.25, 0.3],
    weights=[1/3, 1/3, 1/3],
    n_points=30,
    eps=1e-4,
)

# Show diagnostics
surr.print_summary()

In [ ]:
# Evaluate at a specific point
spots = [100.0, 105.0, 95.0]
price = surr.evaluate(spots)
greeks = surr.greeks(spots)

print(f"Spot prices: {spots}")
print(f"Option price: {price:.4f}")
print(f"Deltas: {[f'{d:.4f}' for d in greeks['delta']]}")
print(f"Gammas: {[f'{g:.6f}' for g in greeks['gamma']]}")

In [ ]:
# Evaluate across a range of spot prices for asset 1
spot_range = np.linspace(80, 120, 100)
prices = [surr.evaluate([s, 100.0, 100.0]) for s in spot_range]
deltas_1 = [surr.greeks([s, 100.0, 100.0])['delta'][0] for s in spot_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(spot_range, prices, 'b-', linewidth=2)
ax1.set_xlabel('Asset 1 Spot Price')
ax1.set_ylabel('Basket Option Price')
ax1.set_title('Price vs Asset 1 Spot (others at 100)')
ax1.grid(alpha=0.3)

ax2.plot(spot_range, deltas_1, 'r-', linewidth=2)
ax2.set_xlabel('Asset 1 Spot Price')
ax2.set_ylabel('Delta (Asset 1)')
ax2.set_title('Delta vs Asset 1 Spot')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Visualization

Use the built-in visualization functions for 2-asset pricing surfaces.

In [ ]:
from tensorquantlib import plot_pricing_surface, plot_greeks_surface, plot_tt_ranks

# Build a 2-asset surrogate for visualization
surr_2d = TTSurrogate.from_basket_analytic(
    S0_ranges=[(80, 120), (80, 120)],
    K=100, T=1.0, r=0.05,
    sigma=[0.2, 0.25],
    weights=[0.5, 0.5],
    n_points=40,
    eps=1e-4,
)

surr_2d.print_summary()

In [ ]:
# Plot the pricing surface — uses surr_2d.plot_surface() convenience method
fig, ax = surr_2d.plot_surface(title="2-Asset Basket Pricing Surface")


In [ ]:
# Plot Greek (Delta) surfaces — uses surr_2d.plot_greeks() convenience method
fig, axes = surr_2d.plot_greeks(title="Delta Surface")


In [ ]:
# Plot TT-rank structure — uses surr_2d.plot_ranks() convenience method
fig, ax = surr_2d.plot_ranks()


---
## 6. TT-Cross for 6+ Assets

The key limitation of TT-SVD is that it requires the **full** pricing grid: `n_points^d` evaluations. For 6 assets with 15 grid points each, that's 15⁶ ≈ 11M evaluations.

**TT-Cross** (Oseledets & Tyrtyshnikov 2010) solves this by sampling the function at **strategically selected** index combinations:
- Only O(d · r² · n) evaluations needed
- 6 assets, 15 pts, rank 10 → ~54,000 evaluations (200× fewer)

Use `TTSurrogate.from_function(fn, axes)` which calls TT-Cross internally.

In [ ]:
import functools
from tensorquantlib import bs_price_numpy, TTSurrogate

# ── 6-asset basket via TT-Cross ──────────────────────────────────────
# Grid: 15 points per asset axis from 80 to 120
n_pts = 15
n_assets = 6
axes_6d = [np.linspace(80.0, 120.0, n_pts)] * n_assets

# Average-basket payoff priced with Black-Scholes (geometric approx)
K_basket = 100.0
T_basket, r_basket, sigma_basket = 1.0, 0.05, 0.2


def basket_pricer_6d(*indices):
    """Evaluate basket call price. Takes n_assets integer grid indices."""
    spots = [axes_6d[k][i] for k, i in enumerate(indices)]
    avg_spot = float(np.mean(spots))
    return float(bs_price_numpy(avg_spot, K_basket, T_basket, r_basket, sigma_basket))


print("Building 6-asset surrogate via TT-Cross...")
print(f"Grid size would be: {n_pts}^{n_assets} = {n_pts**n_assets:,} points")

surr_6d = TTSurrogate.from_function(
    fn=basket_pricer_6d,
    axes=axes_6d,
    eps=1e-3,
    max_rank=8,
    n_sweeps=6,
)
surr_6d.print_summary()

# Evaluate at a spot
test_spots = [100.0] * n_assets
price_surr = surr_6d.evaluate(test_spots)
price_ref = basket_pricer_6d(*[n_pts // 2] * n_assets)  # mid-grid reference
print(f"\nSurrogate price at [100,...,100]: {price_surr:.4f}")
print(f"Reference price at mid-grid:     {price_ref:.4f}")

---
## 7. Stochastic Volatility — Heston Model

The **Heston (1993)** model introduces a mean-reverting stochastic variance process:

$$dS_t = r S_t \, dt + \sqrt{v_t} S_t \, dW_t^S$$
$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW_t^v, \quad \langle dW^S, dW^v \rangle = \rho\,dt$$

TensorQuantLib provides both semi-analytic pricing (via the characteristic function) and Monte Carlo.

In [ ]:
from tensorquantlib import HestonParams, heston_price

# Heston parameters
params = HestonParams(
    kappa=2.0,    # mean-reversion speed
    theta=0.04,   # long-run variance (= 20%² vol)
    xi=0.3,       # vol-of-vol
    rho=-0.7,     # spot-vol correlation
    v0=0.04,      # initial variance
)

S, K, T, r = 100.0, 100.0, 1.0, 0.05

# Semi-analytic price (characteristic function integration)
price_heston = heston_price(S, K, T, r, params)
print(f"Heston call price (semi-analytic): {price_heston:.6f}")

# Compare to flat-vol Black-Scholes at 20%
price_bs = bs_price_numpy(S, K, T, r, sigma=0.20)
print(f"Black-Scholes call price (σ=20%):  {price_bs:.6f}")
print(f"Heston vs BS difference:           {price_heston - price_bs:+.6f}")

# Price across strikes to show the vol smile effect
strikes = np.linspace(80, 120, 41)
prices_h = [heston_price(S, k, T, r, params) for k in strikes]
prices_b = [bs_price_numpy(S, k, T, r, sigma=0.20) for k in strikes]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, prices_h, 'b-', linewidth=2, label='Heston')
ax.plot(strikes, prices_b, 'r--', linewidth=1.5, label='Black-Scholes (σ=20%)')
ax.set_xlabel('Strike K')
ax.set_ylabel('Call Price')
ax.set_title('Heston vs Black-Scholes Call Price')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Volatility Surfaces — SABR and SVI

**SABR** (Hagan et al. 2002) is the industry standard for interest rate options. It provides an analytic formula for implied vol as a function of strike.

**SVI** (Gatheral 2004) is a flexible parameterization for the vol smile slice by slice, widely used for equities.

In [ ]:
from tensorquantlib import sabr_implied_vol, svi_implied_vol, svi_calibrate

S_fwd = 100.0  # forward price
T_vol = 1.0

strikes = np.linspace(80, 120, 41)
log_moneyness = np.log(strikes / S_fwd)

# ── SABR ──────────────────────────────────────────────────────────────
# SABR parameters: alpha (ATM vol level), beta (backbone), rho (skew), nu (vol-of-vol)
alpha, beta, rho_sabr, nu = 0.20, 0.5, -0.3, 0.4
sabr_vols = np.array([sabr_implied_vol(S_fwd, k, T_vol, alpha, beta, rho_sabr, nu) for k in strikes])

# ── SVI with example parameters ──────────────────────────────────────
# SVI raw: w(k) = a + b*(rho*(k-m) + sqrt((k-m)^2 + sigma^2))
# Implied vol = sqrt(w(k) / T)
svi_params = {'a': 0.04, 'b': 0.10, 'rho': -0.3, 'm': 0.0, 'sigma': 0.15}
svi_vols = np.array([svi_implied_vol(k, T_vol, **svi_params) for k in log_moneyness])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, sabr_vols * 100, 'b-', linewidth=2, label='SABR')
ax.plot(strikes, svi_vols * 100, 'r--', linewidth=2, label='SVI')
ax.axhline(20.0, color='gray', linestyle=':', alpha=0.7, label='ATM 20%')
ax.axvline(S_fwd, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Strike K')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('Volatility Smile: SABR vs SVI')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Product Coverage — Exotics, IR, FX, Credit

TensorQuantLib v0.3.0 covers the full derivatives landscape in one library.

In [ ]:
from tensorquantlib import (
    # Exotic options
    asian_price_mc, barrier_price_mc, digital_price,
    # Interest rates
    vasicek_bond_price, cir_bond_price, swap_rate,
    # FX
    garman_kohlhagen,
    # Credit
    merton_default_prob, cds_spread,
)

np.random.seed(42)

print("=" * 55)
print("EXOTIC OPTIONS")
print("=" * 55)
S0 = 100.0; K = 100.0; T = 1.0; r = 0.05; sigma = 0.20

Asian_price = asian_price_mc(S0, K, T, r, sigma, n_paths=50_000)
print(f"Asian call (arithmetic, MC):    {Asian_price:.4f}")

barrier_price = barrier_price_mc(S0, K, T, r, sigma, H=110.0,
                                  barrier_type='up-and-out', n_paths=50_000)
print(f"Up-and-out barrier call (MC):   {barrier_price:.4f}")

digital = digital_price(S0, K, T, r, sigma, payoff=1.0)
print(f"Digital call (analytic):        {digital:.4f}")

print("\n" + "=" * 55)
print("INTEREST RATE MODELS")
print("=" * 55)
kappa, theta_r, xi_r, r0 = 0.5, 0.05, 0.01, 0.03
P_vasicek = vasicek_bond_price(r0, kappa, theta_r, xi_r, T=5.0)
P_cir = cir_bond_price(r0, kappa, theta_r, xi_r, T=5.0)
print(f"Vasicek 5Y bond price: {P_vasicek:.6f}")
print(f"CIR     5Y bond price: {P_cir:.6f}")

print("\n" + "=" * 55)
print("FX OPTIONS")
print("=" * 55)
S_fx, K_fx, r_d, r_f = 1.25, 1.25, 0.03, 0.02
gk_call = garman_kohlhagen(S_fx, K_fx, T, r_d, r_f, sigma=0.12)
print(f"GBP/USD ATM call (Garman-Kohlhagen): {gk_call:.6f}")

print("\n" + "=" * 55)
print("CREDIT RISK")
print("=" * 55)
V0, D, sigma_V, r_rf = 100.0, 80.0, 0.25, 0.05
dp = merton_default_prob(V0, D, sigma_V, r_rf, T=1.0)
cds = cds_spread(hazard_rate=0.02, recovery=0.40, T=5.0)
print(f"Merton 1Y default probability: {dp*100:.2f}%")
print(f"5Y CDS spread (spread bps):    {cds*10000:.1f} bps")

---
## Summary — TensorQuantLib v0.3.0

| Section | Feature | Key Functions |
|---------|---------|--------------|
| **1. Autograd** | Reverse-mode autodiff engine (23 ops) | `Tensor`, `.backward()` |
| **2. Black-Scholes** | Analytic + autograd pricing | `bs_price_tensor`, `bs_delta` |
| **3. Second-Order Greeks** | Gamma/Vanna/Volga via 2nd-order autodiff | `gamma_autograd`, `vanna_autograd`, `volga_autograd` |
| **4. TT-SVD** | Compress pricing tensors, controllable accuracy | `tt_svd`, `tt_round`, `tt_error` |
| **5. TT-Surrogate** | End-to-end basket option surrogate (≤5 assets) | `TTSurrogate.from_basket_analytic` |
| **6. TT-Cross** | 6+ assets without full grid (O(d·r²·n) evals) | `TTSurrogate.from_function`, `tt_cross` |
| **7. Stochastic Vol** | Heston model (semi-analytic + MC) | `HestonParams`, `heston_price` |
| **8. Vol Surfaces** | SABR and SVI parameterizations | `sabr_implied_vol`, `svi_implied_vol` |
| **9. Product Coverage** | Exotics · IR · FX · Credit | `asian_price_mc`, `vasicek_bond_price`, `garman_kohlhagen`, `cds_spread` |
| **10. Visualization** | Publication-ready pricing/Greek plots | `plot_pricing_surface`, `plot_greeks_surface` |

### What's Next?
- See `examples/` for standalone scripts including `black_scholes_greeks.py` and `demo_basket_tt.py`
- See `benchmarks/bench_tt_vs_mc.py` for TT-vs-Monte-Carlo performance comparisons
- See the [project documentation](https://raswanthmalai19.github.io/TensorQuantLib./) for the full API reference